# Appendix：ColabとPC・Google DriveのExcelファイル操作

このノートブックは **Google Colabで実行** します。各コードセルは単独でも実行できます。Mac・Windows共通のコードです。

| やりたいこと | ファイルの移動 | 使う機能 |
|---|---|---|
| PCから読み込む | PC → Colabの `/content/` | `files.upload()` |
| PCへ保存する | Colabの `/content/` → PC | `files.download()` |
| Driveから読み込む | Google Drive → Colabのコード | `drive.mount()` + `pd.read_excel()` |
| Driveへ保存する | Colabのコード → Google Drive | `drive.mount()` + `to_excel()` |

`/content/` は一時領域です。Colabのセッションが終了すると、そこに作ったファイルは残りません。DriveのファイルはDriveに残ります。

**準備**：PC側の読み込みを試すにはExcelファイルが1つ必要です。手元になければ最初のセルで作った練習用ファイルをダウンロードして使えます。Google Driveの操作では本人の認証画面が表示されます。


## 0. 練習用Excelを作成する

まず、Colabの一時領域に練習用のExcelを作ります。PCにもDriveにも、この時点では保存されていません。


In [ ]:
from pathlib import Path
import pandas as pd

work = Path('/content') if Path('/content').exists() else Path.cwd()
sample_path = work / 'appendix_sample_sales.xlsx'
sample = pd.DataFrame({
    '日付': ['2026-09-01', '2026-09-02', '2026-09-03'],
    '店舗': ['鹿児島', '霧島', '鹿児島'],
    '商品': ['茶', '菓子', '茶'],
    '売上金額': [1200, 800, 1500],
})
sample.to_excel(sample_path, index=False)
print('練習用Excelを作成:', sample_path)
sample.head()


## 1. PC上のExcelファイルを読み込む

手元にExcelがなければ、**次のセルで練習用ExcelをPCへダウンロード**します。続くアップロード画面で選びます。どちらのセルも単独で実行できます。MacもWindowsも同じコードです。


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print  # Colab以外のPython環境向け
from pathlib import Path
import pandas as pd

work = Path('/content') if Path('/content').exists() else Path.cwd()
sample_path = work / 'appendix_sample_sales.xlsx'
if not sample_path.exists():
    pd.DataFrame({
        '日付': ['2026-09-01', '2026-09-02', '2026-09-03'],
        '店舗': ['鹿児島', '霧島', '鹿児島'],
        '商品': ['茶', '菓子', '茶'],
        '売上金額': [1200, 800, 1500],
    }).to_excel(sample_path, index=False)
print('ダウンロードするExcel:', sample_path)
display(pd.read_excel(sample_path).head())
try:
    from google.colab import files
    files.download(str(sample_path))
except ImportError:
    print('Colabで実行するとPCへダウンロードできます。')
except Exception as exc:
    print('ダウンロードできませんでした。Colab左側のファイル一覧から取得できます。', type(exc).__name__)


次のセルを実行するとファイル選択画面が開きます。**Excelファイルを1つだけ選択**してください。アップロードしたファイルはColabの一時領域へコピーされます。


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print  # Colab以外のPython環境向け
from pathlib import Path
import pandas as pd

work = Path('/content') if Path('/content').exists() else Path.cwd()
try:
    from google.colab import files
except ImportError:
    print('このセルのファイル選択はGoogle Colabで利用できます。')
else:
    try:
        uploaded = files.upload()
        excel_names = [name for name in uploaded if Path(name).suffix.lower() == '.xlsx']
        if not excel_names:
            print('Excelが選択されませんでした。必要ならこのセルを再実行してください。')
        else:
            local_filename = excel_names[0]
            if len(excel_names) > 1:
                print('複数選択されたため、先頭のExcelを読み込みます:', local_filename)
            local_input = work / local_filename
            local_df = pd.read_excel(local_input)
            print('PCから読み込み:', local_input)
            display(local_df.head())
    except Exception as exc:
        print('Excelを読み込めませんでした。形式を確認して再実行してください。', type(exc).__name__)


## 2. 加工結果をPCへ保存する

PCからの読み込みセルを実行済みであればそのデータを加工します。このセルだけ実行した場合は練習用データを使います。元のExcelは変更しません。


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print  # Colab以外のPython環境向け
from pathlib import Path
import pandas as pd

work = Path('/content') if Path('/content').exists() else Path.cwd()
# PCからの読み込みセルを実行していればそのデータを使用。
# このセルだけ実行した場合は練習用データを使用。
if 'local_df' in globals() and isinstance(local_df, pd.DataFrame):
    source_df = local_df.copy()
    print('先ほどPCから読み込んだデータを使用します。')
else:
    source_df = pd.DataFrame({
        '日付': ['2026-09-01', '2026-09-02', '2026-09-03'],
        '店舗': ['鹿児島', '霧島', '鹿児島'],
        '商品': ['茶', '菓子', '茶'],
        '売上金額': [1200, 800, 1500],
    })
    print('このセルだけ実行したため練習用データを使用します。')
if '売上金額' not in source_df.columns:
    print('「売上金額」列がありません。列名を合わせてから再実行してください。')
else:
    try:
        local_result = source_df.copy()
        local_result['売上金額'] = pd.to_numeric(local_result['売上金額'], errors='raise')
        local_result['税込金額（10%の例）'] = (local_result['売上金額'] * 1.10).round().astype(int)
        local_output = work / 'appendix_pc_result.xlsx'
        local_result.to_excel(local_output, index=False)
        display(local_result.head())
        try:
            from google.colab import files
            files.download(str(local_output))
        except ImportError:
            print('Colabで実行するとPCへダウンロードできます:', local_output)
        except Exception as exc:
            print('自動ダウンロードできませんでした。Colab左側のファイル一覧から取得できます。', type(exc).__name__)
    except (ValueError, TypeError) as exc:
        print('売上金額を数値に変換できません。元データを確認してください。', type(exc).__name__)


## 3. Google Drive上のExcelファイルを読み込む

本人の認証後、**マイドライブ/ColabTips/Appendix** に練習用Excelを作成して読み込みます。認証を完了しなかった場合は理由を表示して終了します。


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print  # Colab以外のPython環境向け
from pathlib import Path
import pandas as pd

try:
    from google.colab import drive
except ImportError:
    print('Google Driveとの接続はGoogle Colabで利用できます。')
else:
    try:
        drive.mount('/content/drive')
        drive_folder = Path('/content/drive/MyDrive/ColabTips/Appendix')
        drive_folder.mkdir(parents=True, exist_ok=True)
        drive_input = drive_folder / 'appendix_sample_sales.xlsx'
        if not drive_input.exists():
            pd.DataFrame({
                '日付': ['2026-09-01', '2026-09-02', '2026-09-03'],
                '店舗': ['鹿児島', '霧島', '鹿児島'],
                '商品': ['茶', '菓子', '茶'],
                '売上金額': [1200, 800, 1500],
            }).to_excel(drive_input, index=False)
            print('Driveに練習用Excelを作成:', drive_input)
        drive_df = pd.read_excel(drive_input)
        print('Driveから読み込んだ内容:')
        display(drive_df.head())
    except Exception as exc:
        print('Driveに接続・読み込みできませんでした。認証とフォルダを確認してください。', type(exc).__name__)


**自分のExcelを読む場合**：`drive_input` に、自分のマイドライブ内のExcelのパスを指定します。例えば `Path('/content/drive/MyDrive/売上/2026年9月.xlsx')` です。Driveの画面に表示されるフォルダ名とパスが一致するか確認してください。


## 4. 加工結果をGoogle Driveへ保存する

このセルだけでもDriveへの接続から実行できます。練習用フォルダに元のExcelとは別名で保存します。認証を完了しなかった場合は理由を表示して終了します。


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print  # Colab以外のPython環境向け
from pathlib import Path
import pandas as pd

try:
    from google.colab import drive
except ImportError:
    print('Google Driveへの保存はGoogle Colabで利用できます。')
else:
    try:
        drive.mount('/content/drive')
        drive_folder = Path('/content/drive/MyDrive/ColabTips/Appendix')
        drive_folder.mkdir(parents=True, exist_ok=True)
        drive_input = drive_folder / 'appendix_sample_sales.xlsx'
        if not drive_input.exists():
            pd.DataFrame({
                '日付': ['2026-09-01', '2026-09-02', '2026-09-03'],
                '店舗': ['鹿児島', '霧島', '鹿児島'],
                '商品': ['茶', '菓子', '茶'],
                '売上金額': [1200, 800, 1500],
            }).to_excel(drive_input, index=False)
        drive_df = pd.read_excel(drive_input)
        if '売上金額' not in drive_df.columns:
            print('「売上金額」列がありません。保存は行いませんでした。')
        else:
            drive_result = drive_df.copy()
            drive_result['売上金額'] = pd.to_numeric(drive_result['売上金額'], errors='raise')
            drive_result['税込金額（10%の例）'] = (drive_result['売上金額'] * 1.10).round().astype(int)
            drive_output = drive_folder / 'appendix_drive_result.xlsx'
            drive_result.to_excel(drive_output, index=False)
            print('Google Driveに保存:', drive_output)
            display(pd.read_excel(drive_output).head())
    except Exception as exc:
        print('Driveに接続・保存できませんでした。認証・列名・フォルダを確認してください。', type(exc).__name__)


## 練習問題と注意

1. 自分で用意したExcelをPCからアップロードし、`head()` で中身を確認する。
2. 売上を店舗別に集計し、PCへダウンロードする。
3. Drive内の自分のファイルのパスを指定して読み込み、別名で保存する。

Excelに複数シート・数式・書式がある場合、`pd.read_excel()` で読み取った表を `to_excel()` すると元のブック全体を保持するわけではありません。元のExcelを直接上書きせず、まず別名で保存して内容を確認してください。
